# 00 — SPIKE: SD3.5 inpaint-edit LoRA feasibility

**Decision gate before building the full inpaint-edit harness.**

Why this exists: SD3.5 has no official inpaint/instruction-edit trainer, and the
SD3 ControlNet-Inpainting is SD3-medium only (NOT 3.5). So we must hand-roll the
edit conditioning. This spike trains ~50 steps on ~20 PIPE pairs to confirm the
conditioning is wired correctly (loss finite + trending down) before investing
in the full pipeline.

PASS  -> proceed to build the full SD3.5 inpaint-edit LoRA trainer.
FAIL  -> rethink the conditioning (adapter design) or reconsider SD3.5 constraint.

## 1. Setup

In [ ]:
import subprocess, sys
from pathlib import Path
REPO = Path('/kaggle/working/VIN')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/BDT-17/VIN.git',str(REPO)], check=True)
sys.path.insert(0, str(REPO))
# Pin a mutually-compatible stack. diffusers 0.31.0 needs transformers <=4.46
# (newer transformers removed FLAX_WEIGHTS_NAME, which breaks the SD3 pipeline import).
!pip install -q 'diffusers==0.31.0' 'transformers==4.46.3' 'tokenizers<0.21' 'huggingface_hub<0.26' 'accelerate>=0.33,<1.1' 'peft>=0.12,<0.14' 'datasets>=2.20' 'safetensors>=0.4.3' 'sentencepiece' 'protobuf' 'pillow>=10' numpy
print('--- verify versions ---')
import importlib
for m in ('diffusers','transformers','huggingface_hub','accelerate','peft'):
    try:
        print(m, importlib.import_module(m).__version__)
    except Exception as e:
        print(m, 'IMPORT ERROR', e)
import torch
assert torch.cuda.is_available(), 'Need GPU'
print('GPU:', torch.cuda.get_device_name(0))
print('\nIMPORTANT: if Kaggle had transformers preinstalled, RESTART the kernel\n'
      'after this cell (Run -> Restart) so the pinned versions take effect, then\n'
      're-run from here.')

## 2. Build ~20 PIPE pairs (source/target/mask)

Reuses the PIPE eval builder — source_img + derived mask + target_img are exactly
the (input, mask, ground-truth) the edit trainer needs.

In [ ]:
from LoRA.data.build_eval_cases_pipe import run_build_pipe_eval
import json
WORK = Path('/kaggle/working/vin_lora')
eval_root = run_build_pipe_eval(WORK, eval_set='spike_pairs', split='test',
                               person_only=True, limit=20)
cases = [json.loads(l) for l in (eval_root/'cases.jsonl').read_text().splitlines() if l.strip()]
pairs = [{
    'source_path': str(eval_root/c['image_path']),
    'target_path': str(eval_root/c['reference_path']),
    'mask_path':   str(eval_root/c['mask_path']),
    'prompt': 'a photo of <vin_ped> pedestrian, ' + (c['prompt_fields'].get('instruction','') or 'a person'),
} for c in cases]
print(len(pairs), 'pairs ready')

## 3. Run the spike (50 steps)

In [ ]:
from LoRA.train.spike_inpaint_edit import run_spike
verdict = run_spike(pairs, base_model_id='stabilityai/stable-diffusion-3.5-medium',
                    steps=50, rank=8, lr=1e-4, resolution=512)
verdict

## 4. Decision

- `loss_dropped == true` and `all_finite == true` -> **PASS**: build the full
  inpaint-edit LoRA trainer (data loader over PIPE-train, this conditioning,
  proper sampling, checkpointing, eval).
- otherwise -> **STOP**: the Conv adapter / conditioning needs rework before
  committing. Report the loss curve back.